# **3. Предобработка данных и хронологическое разбиение (Data Preprocessing and Chronological Splitting): подготовка данных для регрессионного моделирования**

* __Цель предобработки данных:__ сформировать причинно-корректные train, validation и test наборы и обучить преобразования исключительно по обучающей выборке.
* __Задачи предобработки данных:__
  - выполнить хронологическое разбиение без перемешивания;
  - отделить модельные признаки, target и временные метки;
  - применить train-only импутацию, масштабирование и категориальное кодирование;
  - проверить отсутствие служебных колонок и утечки статистик.
* __Алгоритм выполнения:__
  1. Загрузить подготовленные данные и сформировать признаки.
  2. Выбрать группы модельных predictors.
  3. Разделить наблюдения в отношении 70% / 10% / 20%.
  4. Отделить `X`, `y` и timestamps.
  5. Обучить preprocessor на `X_train` и применить его к validation и test.
  6. Выполнить контрольные проверки и интерпретировать результат.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import DATETIME_COLUMN, REPORTS_DIR, TARGET_COLUMN
from traffic_forecasting.data_loader import load_raw_data
from traffic_forecasting.features import build_feature_dataset
from traffic_forecasting.logging_utils import setup_logger
from traffic_forecasting.preprocessing import (
    EXCLUDED_MODEL_COLUMNS,
    get_model_feature_groups,
    prepare_model_inputs,
    split_chronologically,
    transform_model_inputs,
)

logger = setup_logger("notebooks.data_preprocessing")

TABLES_DIR = REPORTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## **3.1. Загрузка данных и формирование признаков (Data Loading and Feature Construction)**

In [ ]:
raw_data = load_raw_data()
feature_data = build_feature_dataset(raw_data)

dataset_shapes = pd.DataFrame(
    {
        "dataset": ["prepared_raw_data", "feature_dataset"],
        "rows": [len(raw_data), len(feature_data)],
        "columns": [raw_data.shape[1], feature_data.shape[1]],
    }
)

display(Markdown("### **Размерности наборов данных (Dataset Shapes)**"))
display(dataset_shapes)

## **3.2. Отбор модельных признаков (Model Feature Selection)**

In [ ]:
model_feature_groups = get_model_feature_groups()
feature_group_summary = pd.DataFrame(
    [
        {
            "feature_group": group_name,
            "feature_count": len(columns),
            "features": ", ".join(columns),
        }
        for group_name, columns in model_feature_groups.items()
    ]
)
feature_group_summary.to_csv(TABLES_DIR / "preprocessing_feature_groups.csv", index=False)

display(Markdown("### **Группы модельных признаков (Model Feature Groups)**"))
display(feature_group_summary)

## **3.3. Хронологическое разбиение выборки (Chronological Dataset Splitting)**

In [ ]:
train_data, validation_data, test_data = split_chronologically(feature_data)

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(split_data),
            "share": len(split_data) / len(feature_data),
            "start": split_data[DATETIME_COLUMN].min(),
            "end": split_data[DATETIME_COLUMN].max(),
        }
        for split_name, split_data in (
            ("train", train_data),
            ("validation", validation_data),
            ("test", test_data),
        )
    ]
)
split_summary.to_csv(TABLES_DIR / "chronological_split_summary.csv", index=False)

display(Markdown("### **Размеры и временные границы выборок (Split Sizes and Ranges)**"))
display(split_summary)

## **3.4. Подготовка X, y и временных меток (Preparation of Features, Target, and Timestamps)**

In [ ]:
(
    X_train,
    X_validation,
    X_test,
    y_train,
    y_validation,
    y_test,
    timestamps_train,
    timestamps_validation,
    timestamps_test,
) = prepare_model_inputs(train_data, validation_data, test_data)

exclusion_audit = pd.DataFrame(
    {
        "column": EXCLUDED_MODEL_COLUMNS,
        "present_in_X_train": [column in X_train.columns for column in EXCLUDED_MODEL_COLUMNS],
        "preserved_separately": [True, True, False],
    }
)

display(Markdown("### **Аудит служебных колонок (Service Column Audit)**"))
display(exclusion_audit)
display(Markdown("### **Пример временных меток и target (Timestamp and Target Sample)**"))
display(pd.DataFrame({DATETIME_COLUMN: timestamps_train.head(), TARGET_COLUMN: y_train.head()}))

## **3.5. Обучение и применение preprocessing transformer (Preprocessor Fitting and Application)**

In [ ]:
(
    X_train_transformed,
    X_validation_transformed,
    X_test_transformed,
    preprocessor,
) = transform_model_inputs(X_train, X_validation, X_test)

transformed_shape_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "input_rows": [len(X_train), len(X_validation), len(X_test)],
        "input_columns": [X_train.shape[1], X_validation.shape[1], X_test.shape[1]],
        "output_rows": [
            X_train_transformed.shape[0],
            X_validation_transformed.shape[0],
            X_test_transformed.shape[0],
        ],
        "output_columns": [
            X_train_transformed.shape[1],
            X_validation_transformed.shape[1],
            X_test_transformed.shape[1],
        ],
    }
)
transformed_shape_summary.to_csv(
    TABLES_DIR / "preprocessing_transformed_shape_summary.csv",
    index=False,
)

display(Markdown("### **Размерности преобразованных матриц (Transformed Matrix Shapes)**"))
display(transformed_shape_summary)

## **3.6. Аудит train-only предобработки (Train-Only Preprocessing Audit)**

In [ ]:
continuous_columns = model_feature_groups["continuous_numeric"]
categorical_columns = model_feature_groups["categorical"]
continuous_transformer = preprocessor.named_transformers_["continuous"]
categorical_transformer = preprocessor.named_transformers_["categorical"]

imputed_train = continuous_transformer.named_steps["imputer"].transform(
    X_train.loc[:, continuous_columns]
)
scaler_means_match = np.allclose(
    continuous_transformer.named_steps["scaler"].mean_,
    imputed_train.mean(axis=0),
)

encoder_categories = categorical_transformer.named_steps["encoder"].categories_
categories_come_from_train = all(
    set(categories).issubset(set(X_train[column].dropna()) | {"unknown"})
    for column, categories in zip(categorical_columns, encoder_categories, strict=True)
)

train_only_audit = pd.DataFrame(
    {
        "check": ["scaler_means_match_imputed_train", "encoder_categories_come_from_train"],
        "passed": [scaler_means_match, categories_come_from_train],
    }
)

display(Markdown("### **Результаты train-only аудита (Train-Only Audit Results)**"))
display(train_only_audit)

## **3.7. Контроль итоговой структуры данных (Final Data Structure Checks)**

In [ ]:
final_checks = pd.DataFrame(
    {
        "check": [
            "train_precedes_validation",
            "validation_precedes_test",
            "train_inputs_are_aligned",
            "validation_inputs_are_aligned",
            "test_inputs_are_aligned",
            "transformed_width_is_consistent",
        ],
        "passed": [
            timestamps_train.max() < timestamps_validation.min(),
            timestamps_validation.max() < timestamps_test.min(),
            len(X_train) == len(y_train) == len(timestamps_train),
            len(X_validation) == len(y_validation) == len(timestamps_validation),
            len(X_test) == len(y_test) == len(timestamps_test),
            X_train_transformed.shape[1]
            == X_validation_transformed.shape[1]
            == X_test_transformed.shape[1],
        ],
    }
)

display(Markdown("### **Итоговые проверки предобработки (Final Preprocessing Checks)**"))
display(final_checks)

In [ ]:
# Explicit notebook validation checks

assert not exclusion_audit["present_in_X_train"].any(), (
    "Service or target columns were found in X_train."
)

assert train_only_audit["passed"].all(), "Train-only preprocessing audit failed."

assert final_checks["passed"].all(), "Final preprocessing checks failed."

logger.info("All explicit preprocessing checks passed.")

In [ ]:
# Save preprocessing audit tables

TABLES_DIR.mkdir(parents=True, exist_ok=True)

exclusion_audit.to_csv(
    TABLES_DIR / "preprocessing_service_column_audit.csv",
    index=False,
)

train_only_audit.to_csv(
    TABLES_DIR / "preprocessing_train_only_audit.csv",
    index=False,
)

final_checks.to_csv(
    TABLES_DIR / "preprocessing_final_checks.csv",
    index=False,
)

logger.info("Preprocessing audit tables saved to %s", TABLES_DIR)

## **3.8. Анализ и интерпретация результатов предварительной обработки данных (Analysis and Interpretation of Data Preprocessing Results)**

На этапе предварительной обработки данных расширенный набор признаков Metro Interstate Traffic Volume был преобразован в модельные входы, пригодные для последующего обучения и сравнения регрессионных моделей прогнозирования транспортной нагрузки. Основное внимание на данном этапе было уделено хронологическому разбиению выборки, отделению целевой переменной, сохранению временных меток и построению preprocessing-конвейера без утечки данных.

**Ключевые результаты:**
1. **Выполнено хронологическое разбиение набора данных.**
   Подготовленный набор признаков был разделен на обучающую, валидационную и тестовую выборки в пропорции 70% / 10% / 20%. Разбиение выполнялось без случайного перемешивания, что позволяет сохранить причинно-следственную структуру временного ряда. Обучающая выборка предшествует валидационной, а валидационная — тестовой.
2. **Сформированы отдельные наборы `X`, `y` и временных меток.**
   Для каждого подмножества были отдельно выделены матрица признаков `X`, целевой вектор `y` и временные метки. Целевая переменная `traffic_volume` была исключена из модельных признаков и сохранена отдельно. Поле `date_time` также не используется как входной признак модели, но сохраняется отдельно для последующего анализа прогнозов и построения временных графиков.
3. **Исключены служебные и исходные немодельные признаки.**
   Из модельной матрицы признаков были исключены `date_time`, `traffic_volume` и исходное поле `holiday`. Признак `holiday` не передается в модель напрямую, поскольку его информация уже представлена в виде бинарного индикатора `is_holiday`.
4. **Построен preprocessing-конвейер для разных групп признаков.**
   Для непрерывных числовых признаков используется медианная импутация и стандартизация. К данной группе относятся погодные числовые признаки, лаговые признаки и rolling-признаки. Категориальные погодные признаки `weather_main` и `weather_description` обрабатываются с помощью импутации константой `unknown` и One-Hot Encoding. Бинарные календарные признаки и циклические sin/cos-признаки передаются без масштабирования.
5. **Обеспечена защита от утечки статистики validation и test.**
   Параметры preprocessing-преобразований рассчитываются только на обучающей выборке. Валидационная и тестовая выборки не участвуют в обучении `SimpleImputer`, `StandardScaler` и `OneHotEncoder`, а только преобразуются с использованием уже обученного preprocessing transformer.
6. **Проверена согласованность преобразованных модельных входов.**
   Итоговые проверки подтвердили, что границы train / validation / test не пересекаются, порядок временных интервалов сохранен, служебные поля отсутствуют в `X`, а преобразованные матрицы train, validation и test имеют согласованную размерность.

**Итоговое методологическое резюме:** этап Data Preprocessing and Chronological Splitting сформировал корректные модельные входы без нарушения временной структуры данных. Полученные `X`, `y` и timestamps сохраняют разделение прошлого и будущего, а preprocessing-конвейер обучается только на train-выборке.